In [86]:
import json
import os
from datetime import datetime, timedelta
from dataclasses import dataclass
from typing import Optional
import math
import time
from collections import deque
from decimal import Decimal


import numpy as np
import pandas as pd
import talib as ta
import mplfinance as mpf
from dotenv import load_dotenv
from okx.MarketData import MarketAPI
from pandas import DataFrame
from tqdm import tqdm

from IPython.core.interactiveshell import InteractiveShell

InteractiveShell.ast_node_interactivity = 'all'

load_dotenv()
KEY = os.getenv("OKX_API_KEY")
SECRET = os.getenv("OKX_API_SECRET")
assert KEY and SECRET, "API key and secret are required"
market = MarketAPI(KEY, SECRET, flag="0", debug=False)

def to_candles(data: list) -> DataFrame:
    """将数据转换为用于绘制 K 线图的 DataFrame

    Args:
        data: JSON 数据

    Returns:
        DataFrame: 一组 K 线数据帧
    """
    df = pd.DataFrame(
        data,
        columns=[
            "ts",
            "open",
            "high",
            "low",
            "close",
            "volume",
            "volCcy",
            "volCcyQuote",
            "_",
        ],
    )
    return df

def get_current_candlestick(instId: str, bar: str = "1H", limit: int = 100) -> DataFrame:
    """获取当前 K 线数据"""
    if not isinstance(instId, str):
        raise TypeError("instId must be a string")

    # 获取最近 3 个小时级别的 K 线数据
    resp = market.get_candlesticks(instId, bar=bar, limit=100)
    data = resp["data"]
    df = to_candles(data)
    return df

True

In [88]:
import os
import smtplib
from email.mime.text import MIMEText

def send_email(subject: str, text: str, to_email: Optional[str] = None):
    # 邮件发送者和接收者
    from_email = os.getenv("FROM_EMAIL")
    to_email = to_email or os.getenv("TO_EMAIL")
    
    # SMTP 服务器配置
    smtp_server = os.getenv("SMTP_SERVER")
    smtp_port = 587  # 对于 TLS
    smtp_user = from_email
    smtp_password = os.getenv("SMTP_PASSWORD")
    
    if not all([from_email, to_email, smtp_server, smtp_user, smtp_password]):
        print("请配置邮件发送参数")
        return

    # 创建邮件对象
    msg = MIMEText(text, "plain", "utf-8")
    msg["From"] = from_email
    msg["To"] = to_email
    msg["Subject"] = subject

    server = None
    try:
        server = smtplib.SMTP(smtp_server, smtp_port)
        server.starttls()  # 启动TLS加密
        server.login(smtp_user, smtp_password)  # 登录SMTP服务器
        server.sendmail(from_email, to_email, msg.as_string())
        print("邮件发送成功")
    except Exception as e:
        print(f"邮件发送失败: {e}")
    finally:
        if server:
            try:
                server.quit()
            except Exception as e:
                print(f"关闭SMTP连接失败: {e}")

TypeError: send_email() missing 2 required positional arguments: 'subject' and 'text'

In [57]:
# 获取过去一天的数据
def get_df(instId, bar="15m"):
    df = get_current_candlestick(instId, bar=bar, limit=10)
    df["ts"] = df["ts"].astype(int)
    df["ts"] = pd.to_datetime(df["ts"], unit="ms") + timedelta(hours=8)
    # 按时间升序排列
    df = df.sort_values("ts", ascending=True)
    df.reset_index(drop=True, inplace=True)
    df["high"] = df["high"].apply(lambda x: Decimal(x))
    df["low"] = df["low"].apply(lambda x: Decimal(x))
    df["close"] = df["close"].apply(lambda x: Decimal(x))
    df["open"] = df["open"].apply(lambda x: Decimal(x))
    return df

In [69]:
def get_current_change(instId, bar="1H"):
    """获取当前价格变动幅度, 即涨跌幅"""
    df = get_df(instId, bar)
    df["change"] = df["close"].pct_change() * 100
    df["change"] = df["change"].apply(lambda x: round(x, 2))
    df["change"] = df["change"].fillna(0)
    if df["change"].empty:
        return None
    return df

In [70]:
change = get_current_change("PEOPLE-USDT-SWAP")
change

,ts,open,high,low,close,volume,volCcy,volCcyQuote,_,change
0,2024-06-06 07:00:00,0.11891,0.12457,0.11875,0.11983,4757410,475741000,57989752.322,1,0
1,2024-06-06 08:00:00,0.1198,0.12033,0.11532,0.11625,3476226,347622600,40768113.305,1,-2.99
2,2024-06-06 09:00:00,0.11626,0.12204,0.11552,0.12105,2765428,276542800,33065811.814,1,4.13
3,2024-06-06 10:00:00,0.12106,0.12667,0.12058,0.12415,4581326,458132600,56767203.729,1,2.56
4,2024-06-06 11:00:00,0.12415,0.12533,0.1225,0.12306,2022077,202207700,25029424.94,1,-0.88
...,...,...,...,...,...,...,...,...,...,...
95,2024-06-10 06:00:00,0.12448,0.12758,0.12389,0.12748,1227319,122731900,15482781.936,1,2.48
96,2024-06-10 07:00:00,0.12747,0.1275,0.12525,0.12665,659642,65964200,8331374.541,1,-0.65
97,2024-06-10 08:00:00,0.12666,0.13178,0.12627,0.12915,2323560,232356000,30010814.808,1,1.97
98,2024-06-10 09:00:00,0.12916,0.12924,0.12478,0.12615,2153844,215384400,27311221.135,1,-2.32


In [71]:
def filter_tickers(_type="", bar="1H", threshold=5):
    """过滤出 1 小时涨幅超过指定涨幅的交易对"""
    if _type == "swap":
        with open("./swap_tickers.json", "r") as f:
            tickers = json.load(f)
    elif _type == "spot":
        with open("./spot_tickers.json", "r") as f:
            tickers = json.load(f)
    else:
        raise ValueError("Invalid type")

    results = []
    for item in tickers:
        ticker = item["instId"]
        df = get_current_change(ticker, bar)
        if df is None or len(df) < 2:
            continue
        df["change"] = df["change"].astype(float)
        # 最近两根 K 线的涨幅
        change = max(df["change"].iloc[-1], df["change"].iloc[-2])
        if change > threshold:
            results.append((ticker, change))
    return results

In [76]:
# tickers = filter_tickers(_type="swap", bar="1H", threshold=2.5)
# tickers

[('LPT-USDT-SWAP', 3.22)]

Failed to send email: please run connect() first


SMTPServerDisconnected: please run connect() first

In [91]:
last_result = []
while 1:
    tickers = filter_tickers(_type="swap", bar="1H", threshold=2.5)
    if tickers and tickers != last_result:
        # 通知邮件
        print(tickers)
        text = ""
        for ticker, change in tickers:
            text += f"{ticker}: {change}%\n"
        send_email("OKX 交易对涨幅提醒", text)
        last_result = tickers
    time.sleep(60)

[('MEW-USDT-SWAP', 2.68)]
邮件发送成功
[('MEW-USDT-SWAP', 3.17)]
邮件发送成功
[('LDO-USDT-SWAP', 2.82), ('MEW-USDT-SWAP', 2.81)]
邮件发送成功
[('LDO-USDT-SWAP', 2.61), ('CEL-USDT-SWAP', 3.04)]
邮件发送成功
[('LDO-USDT-SWAP', 2.71), ('CEL-USDT-SWAP', 3.03)]
邮件发送成功
[('LDO-USDT-SWAP', 2.55), ('CEL-USDT-SWAP', 2.92)]
邮件发送成功
[('LDO-USDT-SWAP', 2.61), ('CEL-USDT-SWAP', 3.01)]
邮件发送成功
[('NOT-USDT-SWAP', 2.51)]
邮件发送成功
[('NOT-USDT-SWAP', 2.7)]
邮件发送成功
[('STORJ-USDT-SWAP', 3.19)]
邮件发送成功
[('STORJ-USDT-SWAP', 2.76)]
邮件发送成功
[('STORJ-USDT-SWAP', 2.78)]
邮件发送成功
[('STORJ-USDT-SWAP', 2.74)]
邮件发送成功
[('NOT-USDT-SWAP', 2.64)]
邮件发送成功


ReadTimeout: The read operation timed out